In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# การ transpile ด้วย pass managers

<details>
<summary><b>เวอร์ชันแพ็คเกจ</b></summary>

โค้ดในหน้านี้พัฒนาโดยใช้ requirements ต่อไปนี้
แนะนำให้ใช้เวอร์ชันเหล่านี้หรือใหม่กว่า

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
วิธีที่แนะนำในการ transpile Circuit คือการสร้าง staged pass manager และเรียกใช้เมธอด `run` โดยส่ง circuit เป็น input หน้านี้อธิบายวิธี transpile quantum circuits ด้วยแนวทางนี้
## pass manager (แบบ staged) คืออะไร?
ในบริบทของ Qiskit SDK การ transpilation หมายถึงกระบวนการแปลง circuit ต้นฉบับให้อยู่ในรูปแบบที่เหมาะสมสำหรับการรันบนอุปกรณ์ควอนตัม โดยทั่วไปการ transpilation จะเกิดขึ้นเป็นลำดับขั้นตอนที่เรียกว่า transpiler passes โดย circuit จะถูกประมวลผลโดย transpiler pass แต่ละตัวตามลำดับ และผลลัพธ์ของ pass หนึ่งจะกลายเป็น input ของ pass ถัดไป ตัวอย่างเช่น pass หนึ่งอาจวิเคราะห์ circuit และรวม single-qubit gates ที่ต่อเนื่องกันทั้งหมด จากนั้น pass ถัดไปอาจสังเคราะห์ gates เหล่านี้ให้อยู่ใน basis set ของอุปกรณ์เป้าหมาย transpiler passes ที่มาพร้อมกับ Qiskit อยู่ในโมดูล [qiskit.transpiler.passes](https://docs.quantum.ibm.com/api/qiskit/transpiler_passes)

pass manager คือออบเจ็กต์ที่เก็บรายการ transpiler passes และสามารถรันบน circuit ได้ สร้าง pass manager โดยสร้าง [`PassManager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager) พร้อมรายการ transpiler passes เพื่อรันการ transpilation บน circuit ให้เรียกเมธอด [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager#run) โดยส่ง circuit เป็น input

staged pass manager คือ pass manager ชนิดพิเศษที่แสดงถึงระดับของ abstraction ที่สูงกว่า pass manager ธรรมดา ในขณะที่ pass manager ธรรมดาประกอบด้วย transpiler passes หลายตัว staged pass manager ประกอบด้วย *pass managers* หลายตัว นี่เป็น abstraction ที่มีประโยชน์เพราะการ transpilation มักเกิดขึ้นในขั้นตอนที่ชัดเจน ตามที่อธิบายใน [Transpiler stages](transpiler-stages) โดยแต่ละขั้นตอนจะแสดงด้วย pass manager staged pass managers แสดงด้วยคลาส [`StagedPassManager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.StagedPassManager) ส่วนที่เหลือของหน้านี้อธิบายวิธีสร้างและปรับแต่ง (staged) pass managers
## สร้าง preset staged pass manager
เพื่อสร้าง preset staged pass manager ที่มีค่าเริ่มต้นที่เหมาะสม ให้ใช้ฟังก์ชัน [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager):

In [1]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.backend("ibm_fez")
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

เพื่อ transpile circuit หรือรายการของ circuits ด้วย pass manager ให้ส่ง circuit หรือรายการของ circuits ไปยังเมธอด `run` มาลองทำกับ two-qubit circuit ที่ประกอบด้วย Hadamard ตามด้วย CX gates ที่อยู่ติดกันสองตัว:

In [2]:
from qiskit import QuantumRegister, QuantumCircuit

# Create a circuit
qubits = QuantumRegister(2, name="q")
circuit = QuantumCircuit(qubits)
a, b = qubits
circuit.h(a)
circuit.cx(a, b)
circuit.cx(b, a)

# Transpile it by calling the run method of the pass manager
transpiled = pass_manager.run(circuit)

# Draw it, excluding idle qubits from the diagram
transpiled.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/dcc69b72-e13b-4df6-a51f-a5ef2108bae7-0.svg" alt="Output of the previous code cell" />

![ผลลัพธ์ของโค้ดเซลล์ก่อนหน้า](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/dcc69b72-e13b-4df6-a51f-a5ef2108bae7-0.svg)

ดู [Transpilation defaults and configuration options](defaults-and-configuration-options) สำหรับคำอธิบายของ arguments ที่เป็นไปได้สำหรับฟังก์ชัน `generate_preset_pass_manager` arguments ของ `generate_preset_pass_manager` ตรงกับ arguments ของฟังก์ชัน [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile)

<CodeAssistantAdmonition tagLine="Having trouble remembering pass manager details? Try asking Qiskit Code Assistant." />

ถ้า preset pass managers ไม่ตอบสนองความต้องการ สามารถปรับแต่งการ transpilation ได้โดยสร้าง (staged) pass managers หรือแม้แต่ transpilation passes ส่วนที่เหลือของหน้านี้อธิบายวิธีสร้าง pass managers สำหรับคำแนะนำในการสร้าง transpilation passes ดู [Write your own transpiler pass](custom-transpiler-pass)

## สร้าง pass manager เอง

โมดูล [qiskit.transpiler.passes](https://docs.quantum.ibm.com/api/qiskit/transpiler_passes) มี transpiler passes จำนวนมากที่สามารถใช้สร้าง pass managers ได้ เพื่อสร้าง pass manager ให้สร้าง `PassManager` พร้อมรายการ passes ตัวอย่างเช่น โค้ดต่อไปนี้สร้าง transpiler pass ที่รวม adjacent two-qubit gates และสังเคราะห์ให้อยู่ใน basis ของ Gate [$R_y$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RYGate), [$R_z$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RZGate) และ [$R_{xx}$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RXXGate)

In [3]:
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import (
    Collect2qBlocks,
    ConsolidateBlocks,
    UnitarySynthesis,
)

basis_gates = ["rx", "ry", "rxx"]
translate = PassManager(
    [
        Collect2qBlocks(),
        ConsolidateBlocks(basis_gates=basis_gates),
        UnitarySynthesis(basis_gates),
    ]
)

เพื่อแสดงให้เห็น pass manager นี้ในทางปฏิบัติ ให้ทดสอบกับ two-qubit circuit ที่ประกอบด้วย Hadamard ตามด้วย CX gates ที่อยู่ติดกันสองตัว:

In [4]:
from qiskit import QuantumRegister, QuantumCircuit

qubits = QuantumRegister(2, name="q")
circuit = QuantumCircuit(qubits)

a, b = qubits
circuit.h(a)
circuit.cx(a, b)
circuit.cx(b, a)

circuit.draw("mpl")

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/bc208935-e63c-461b-90d0-a6f4cabc16b6-0.svg" alt="Output of the previous code cell" />

![ผลลัพธ์ของโค้ดเซลล์ก่อนหน้า](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/bc208935-e63c-461b-90d0-a6f4cabc16b6-0.svg)

เพื่อรัน pass manager บน circuit ให้เรียกเมธอด `run`

In [5]:
translated = translate.run(circuit)
translated.draw("mpl")

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/adb5c242-5cba-4878-a00d-4ec47737d029-0.svg" alt="Output of the previous code cell" />

![ผลลัพธ์ของโค้ดเซลล์ก่อนหน้า](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/adb5c242-5cba-4878-a00d-4ec47737d029-0.svg)

สำหรับตัวอย่างขั้นสูงที่แสดงวิธีสร้าง pass manager เพื่อใช้เทคนิคการระงับข้อผิดพลาดที่เรียกว่า dynamical decoupling ดู [Create a pass manager for dynamical decoupling](dynamical-decoupling-pass-manager)

## สร้าง staged pass manager

`StagedPassManager` คือ pass manager ที่ประกอบด้วย stages แต่ละ stage ซึ่งกำหนดโดย instance ของ `PassManager` สามารถสร้าง `StagedPassManager` โดยระบุ stages ที่ต้องการ ตัวอย่างเช่น โค้ดต่อไปนี้สร้าง staged pass manager ที่มีสอง stages คือ `init` และ `translation` โดย `translation` stage กำหนดด้วย pass manager ที่สร้างขึ้นก่อนหน้า

In [6]:
from qiskit.transpiler import PassManager, StagedPassManager
from qiskit.transpiler.passes import UnitarySynthesis, Unroll3qOrMore

basis_gates = ["rx", "ry", "rxx"]
init = PassManager(
    [UnitarySynthesis(basis_gates, min_qubits=3), Unroll3qOrMore()]
)
staged_pm = StagedPassManager(
    stages=["init", "translation"], init=init, translation=translate
)

ไม่มีข้อจำกัดสำหรับจำนวน stages ที่ใส่ใน staged pass manager

อีกวิธีที่มีประโยชน์ในการสร้าง staged pass manager คือเริ่มต้นจาก preset staged pass manager แล้วสลับบาง stages ออก ตัวอย่างเช่น โค้ดต่อไปนี้สร้าง preset pass manager ที่มี optimization level 3 จากนั้นระบุ `pre_layout` stage แบบกำหนดเอง

In [7]:
import numpy as np
from qiskit.circuit.library import HGate, PhaseGate, RXGate, TdgGate, TGate
from qiskit.transpiler.passes import InverseCancellation

pass_manager = generate_preset_pass_manager(3, backend)
inverse_gate_list = [
    HGate(),
    (RXGate(np.pi / 4), RXGate(-np.pi / 4)),
    (PhaseGate(np.pi / 4), PhaseGate(-np.pi / 4)),
    (TGate(), TdgGate()),
]
logical_opt = PassManager(
    [
        InverseCancellation(inverse_gate_list),
    ]
)

# Add pre-layout stage to run extra logical optimization
pass_manager.pre_layout = logical_opt

[ฟังก์ชัน stage generator](https://docs.quantum.ibm.com/api/qiskit/transpiler_preset#stage-generator-functions) อาจมีประโยชน์สำหรับการสร้าง pass managers แบบกำหนดเอง
ฟังก์ชันเหล่านี้สร้าง stages ที่ให้ฟังก์ชันการทำงานทั่วไปที่ใช้ใน pass managers จำนวนมาก
ตัวอย่างเช่น [`generate_embed_passmanager`](https://docs.quantum.ibm.com/api/qiskit/transpiler_preset#qiskit.transpiler.preset_passmanagers.generate_embed_passmanager) สามารถใช้สร้าง stage
เพื่อ "embed" `Layout` เริ่มต้นที่เลือกจาก layout pass ไปยังอุปกรณ์เป้าหมายที่ระบุ

## ขั้นตอนถัดไป
> **Tip:** - [เขียน custom transpiler pass](custom-transpiler-pass)
>     - [สร้าง pass manager สำหรับ dynamical decoupling](dynamical-decoupling-pass-manager)
>     - เพื่อเรียนรู้เพิ่มเติมเกี่ยวกับฟังก์ชัน `generate_preset_passmanager` ดูหัวข้อ [Transpilation default settings and configuration options](defaults-and-configuration-options)
>     - ลองดู [Compare transpiler settings](/guides/circuit-transpilation-settings)
>     - ทบทวน [transpiler API documentation](https://docs.quantum.ibm.com/api/qiskit/transpiler)